# Notebook 04: model hyperparamenter tuning

## 01 Notebook Purpose

This notebook tunes and evaluates the selected Random Forest baseline model for binary prediction of machine failure. It uses the prepared train-test datasets generated in Notebook 02 and builds directly on the baseline modeling results from Notebook 03, where Random Forest was identified as the strongest overall model. Because the target remains strongly imbalanced, hyperparameter tuning is carried out using cross-validation and with particular attention to Average Precision, while final evaluation also considers precision, recall, F1-score, ROC-AUC, and Average Precision. The goal is to determine whether controlled tuning leads to a meaningful improvement over the original baseline model.

## 02. Imports and Load Data

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [3]:
# Load prepared datasets from Notebook 02
X_train = pd.read_csv("../data/processed/X_train_prepared.csv")
X_test = pd.read_csv("../data/processed/X_test_prepared.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8000, 10)
X_test shape: (2000, 10)
y_train shape: (8000,)
y_test shape: (2000,)


In [12]:
display(X_train.head())
display(y_train.head())


,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temperature difference [K],num__ToolWear_x_Torque,cat__Type_H,cat__Type_L,cat__Type_M
0,0.998914,0.604282,-0.460607,0.718305,-0.843997,-1.100842,-0.621945,0.0,0.0,1.0
1,-1.505194,-1.153260,-0.775574,0.638456,0.382263,1.299658,0.646140,0.0,0.0,1.0
2,0.498092,1.077466,-1.007654,0.558607,0.460870,0.599512,0.689544,0.0,0.0,1.0
3,-0.553633,-0.139294,-0.709265,1.626586,-0.372359,0.899575,0.151247,0.0,1.0,0.0
4,-1.455112,-1.018064,1.070019,-1.128202,-0.906882,1.399679,-1.016909,0.0,1.0,0.0


0    0
1    0
2    0
3    0
4    0
Name: Machine failure, dtype: int64

## 03. Baseline Reference Setup

In [6]:
#Recreate baseline model
rf_baseline = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

The baseline Random Forest model is recreated here using the same configuration as in Notebook 03. The setting `class_weight="balanced"` is retained because the target variable is strongly imbalanced, which means the minority failure class should receive greater importance during training. This keeps Notebook 04 directly comparable to the previous baseline modeling step.

In [9]:
#Cross-validation setup and scoring
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision"
}

Stratified 5-fold cross-validation is used so that each fold preserves the original class distribution as closely as possible, which is especially important in an imbalanced classification problem. Multiple scoring metrics are included because accuracy alone would not provide a reliable picture of performance for rare failure events.

In [14]:
#Cross validation
baseline_cv = cross_validate(
    rf_baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

#Results into a dataframe
baseline_cv_results = pd.DataFrame({
    metric: baseline_cv[f"test_{metric}"] for metric in scoring.keys()
})

#Results aggregation
baseline_cv_summary = baseline_cv_results.agg(["mean", "std"]).T
baseline_cv_summary

,mean,std
accuracy,0.987375,0.001556
precision,0.966808,0.022846
recall,0.649562,0.032853
f1,0.776841,0.029112
roc_auc,0.970823,0.013965
avg_precision,0.835380,0.030934


## 04. Hyperparameter Tuning

In [15]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

A moderate hyperparameter search space is defined for the Random Forest model. The goal is not to search exhaustively, but to explore a reasonable range of settings that may improve model performance while keeping the process efficient and credible. The selected parameters affect tree complexity, split behavior, feature selection, and sampling strategy.

In [16]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'bootstrap': [True, False], 'max_depth': [None, 5, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,n_iter,30
,scoring,'average_precision'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


RandomizedSearchCV is used to tune the Random Forest model. This approach samples a fixed number of hyperparameter combinations from the defined search space and evaluates them using cross-validation. The optimization target is Average Precision, which is a suitable metric for this imbalanced classification problem because it focuses more directly on the model’s ability to rank positive failure cases well.

In [17]:
print("Best parameters:")
print(random_search.best_params_)

print("\nBest cross-validated Average Precision:")
print(round(random_search.best_score_, 4))

Best parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': 20, 'bootstrap': True}

Best cross-validated Average Precision:
0.8464


This output shows the best hyperparameter combination found during the randomized search, along with the corresponding cross-validated Average Precision score. These results indicate which Random Forest configuration performed best according to the selected optimization criterion.

In [18]:
rf_tuned = random_search.best_estimator_
rf_tuned

,n_estimators,200
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


The best-performing model identified during hyperparameter tuning is stored as the tuned Random Forest estimator. This model will now be used for final evaluation on the held-out test set.

In [19]:
y_pred_tuned = rf_tuned.predict(X_test)
y_proba_tuned = rf_tuned.predict_proba(X_test)[:, 1]

tuned_test_results = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(y_test, y_pred_tuned),
    "Recall": recall_score(y_test, y_pred_tuned),
    "F1-score": f1_score(y_test, y_pred_tuned),
    "ROC-AUC": roc_auc_score(y_test, y_proba_tuned),
    "Average Precision": average_precision_score(y_test, y_proba_tuned)
}

pd.Series(tuned_test_results).round(4)

Accuracy             0.9910
Precision            0.8906
Recall               0.8382
F1-score             0.8636
ROC-AUC              0.9688
Average Precision    0.8761
dtype: float64

The tuned Random Forest model is evaluated on the held-out test set, which has not been used during cross-validation or hyperparameter tuning. This provides an unbiased estimate of how well the tuned model generalizes to unseen data. 

In [20]:
#Generate Confusion Matrix
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
cm_tuned

array([[1925,    7],
       [  11,   57]])

In [22]:
#Displaying Confusion Matrix values

tn, fp, fn, tp = cm_tuned.ravel()

print("Tuned Random Forest Confusion Matrix")
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives : {tp}")

Tuned Random Forest Confusion Matrix
True Negatives : 1925
False Positives: 7
False Negatives: 11
True Positives : 57


In [23]:
print("Classification report - Tuned Random Forest\n")
print(classification_report(y_test, y_pred_tuned))

Classification report - Tuned Random Forest

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1932
           1       0.89      0.84      0.86        68

    accuracy                           0.99      2000
   macro avg       0.94      0.92      0.93      2000
weighted avg       0.99      0.99      0.99      2000



## 05. Baseline vs. Tuned Comparison

To make the comparison explicit within this notebook, the baseline Random Forest is fitted again on the training data and evaluated on the same test set. This ensures that the tuned model is compared directly against the original baseline under identical conditions.

In [24]:
rf_baseline.fit(X_train, y_train)

y_pred_baseline = rf_baseline.predict(X_test)
y_proba_baseline = rf_baseline.predict_proba(X_test)[:, 1]

baseline_test_results = {
    "Accuracy": accuracy_score(y_test, y_pred_baseline),
    "Precision": precision_score(y_test, y_pred_baseline),
    "Recall": recall_score(y_test, y_pred_baseline),
    "F1-score": f1_score(y_test, y_pred_baseline),
    "ROC-AUC": roc_auc_score(y_test, y_proba_baseline),
    "Average Precision": average_precision_score(y_test, y_proba_baseline)
}

pd.Series(baseline_test_results).round(4)

Accuracy             0.9885
Precision            0.9592
Recall               0.6912
F1-score             0.8034
ROC-AUC              0.9722
Average Precision    0.8627
dtype: float64

In [25]:
comparison_df = pd.DataFrame({
    "Baseline RF": baseline_test_results,
    "Tuned RF": tuned_test_results
}).round(4)

comparison_df["Delta (Tuned - Baseline)"] = (
    comparison_df["Tuned RF"] - comparison_df["Baseline RF"]
).round(4)

comparison_df

,Baseline RF,Tuned RF,Delta (Tuned - Baseline)
Accuracy,0.9885,0.9910,0.0025
Precision,0.9592,0.8906,-0.0686
Recall,0.6912,0.8382,0.1470
F1-score,0.8034,0.8636,0.0602
ROC-AUC,0.9722,0.9688,-0.0034
Average Precision,0.8627,0.8761,0.0134


The tuned Random Forest outperformed the baseline overall, with the most important gain being in **recall**, which increased from **0.6912** to **0.8382**. This means the tuned model detects a substantially larger share of true machine failures, which is particularly valuable in a predictive maintenance setting. The stronger recall also contributed to an improved **F1-score** of **0.8636**, compared with **0.8034** for the baseline.

The tuned model also achieved a small improvement in **Average Precision** and **accuracy**, indicating that tuning improved performance in a meaningful way. However, this came with a trade-off: **precision** decreased from **0.9592** to **0.8906**, meaning the model now produces more false positives than before. **ROC-AUC** also decreased slightly, though only marginally.

Overall, the tuned model appears to offer a better balance for this task, because it identifies more real failures while still maintaining strong overall classification performance.

## 06. Conclusion

In this notebook, the Random Forest baseline selected in Notebook 03 was refined through hyperparameter tuning using cross-validated **Average Precision** as the optimization target.

The tuned model showed a **meaningful improvement over the baseline**, particularly in its ability to detect machine failures. Recall increased from **0.6912** to **0.8382**, which means that the model now captures a substantially larger share of true failures. This improvement also raised the **F1-score** from **0.8034** to **0.8636**, indicating a better overall balance between precision and recall.

The tuned model also slightly improved **Average Precision** and **accuracy**, while **precision** and **ROC-AUC** decreased moderately. This suggests that the tuned model is less conservative than the baseline, but more effective at identifying failure cases, which is a valuable trade-off in a predictive maintenance setting.

Overall, the tuned Random Forest is the stronger model at this stage of the project. A logical next step would be to examine **decision-threshold adjustment** in order to further analyze the trade-off between false positives and false negatives.